# Walkthrough: the 2018 trade war and local innovation

This notebook walks through the main steps of the dissertation using the package in `src/tariff_innovation`.
It reads the processed data built by stages 1–2 of the pipeline, so run `python run_all.py --to 2` (or `make build`) first.

**Design in one line:** regress the county-level change in patents per 100k residents on the county's exposure
to the 2018 tariff shocks, $Z_i = \sum_n s_{in} g_n$. The estimate comes from the equivalent *shock-level* regression
of Borusyak, Hull and Jaravel (2022), which gives exposure-robust standard errors.

In [1]:
import pandas as pd
from tariff_innovation import config
from tariff_innovation.analysis import regressions as reg, shiftshare

pd.set_option("display.precision", 3)

## 1. From counties to industries

Each county has a patent outcome, a set of 2016 controls (GOP vote share and its square, manufacturing, college,
unemployment) and a vector of NAICS-6 employment shares. `ssaggregate` residualises the outcome on the county
controls and turns it into exposure-weighted industry averages with importance weights $s_n$.

In [2]:
county_panel = pd.read_parquet(config.COUNTY_PANEL)
industries = pd.read_parquet(config.INDUSTRY_MAIN)
print(f"{len(county_panel)} counties -> {len(industries)} tradable NAICS-6 industries")
industries[["naics", "s_n", config.OUTCOME_POST, "tariff_shock", "retaliation_shock", *config.INDUSTRY_CONTROLS]].head()

2290 counties -> 333 tradable NAICS-6 industries


,naics,s_n,delta_post_shock_per_100k,tariff_shock,retaliation_shock,markups,prode_share,cap_vadd_ratio,log_real_wage,equip_share
0,311111,2.062e-04,8.692,0.019,4.990e-04,1.81,0.777,0.582,10.913,0.753
1,311119,2.665e-04,0.151,0.015,5.726e-03,1.81,0.670,0.797,11.080,0.632
2,311211,1.161e-04,7.078,0.005,9.516e-04,1.81,0.697,0.787,11.159,0.648
3,311212,3.496e-05,1.965,0.002,6.418e-03,1.81,0.755,0.966,10.820,0.697
4,311213,9.554e-06,14.483,0.000,0.000e+00,1.81,0.667,1.264,11.173,0.636


## 2. Is there enough shock variation?

The effective number of shocks $1/\sum_n s_n^2$ should be large for shock-level asymptotics to apply.

In [3]:
pd.Series(shiftshare.shock_diagnostics(industries, "tariff_shock"))

n_industries     333.000
inverse_hhi      137.152
weighted_mean      0.021
weighted_sd        0.023
max_weight         0.035
dtype: float64

## 3. Balance: were tariffs as-good-as-random?

Each baseline industry characteristic is regressed on the tariff shock. Tariffs fall on **low-markup** industries,
so markups are included as a control in every specification.

In [4]:
pd.DataFrame({
    name: {"coef": fit.params["tariff_shock"], "se": fit.bse["tariff_shock"], "p": fit.pvalues["tariff_shock"]}
    for name, fit in reg.balance_tests(industries).items()
}).T

,coef,se,p
Markups,-3.248,1.413,0.022
Labor,0.204,0.651,0.754
Capital,0.307,1.979,0.877
Wages,-0.613,2.087,0.769
High-Tech,0.243,0.489,0.619


## 4. First stage: do statutory tariffs predict imports?

Barely. Trade diversion to untargeted countries washes out the effect at the NAICS-6 level, which is why the
thesis uses the reduced form.

In [5]:
fs = reg.first_stage(industries, pd.read_parquet(config.IMPORT_CHANGE))
fs.params["tariff_shock"], fs.pvalues["tariff_shock"]

(np.float64(-1.0071022275894972), np.float64(0.11721620332390817))

## 5. Main results

In [6]:
rows = {}
for name, fit, shock in [
    ("U.S. tariff", reg.main_regression(industries), "tariff_shock"),
    ("Retaliation", reg.retaliation_regression(industries), "retaliation_shock"),
]:
    rows[name] = {"coef": fit.params[shock], "se": fit.bse[shock], "p": fit.pvalues[shock], "N": int(fit.nobs)}
pd.DataFrame(rows).T

,coef,se,p,N
U.S. tariff,3.125,42.233,0.941,333.0
Retaliation,37.937,33.638,0.259,333.0


## 6. Inference with null-imposed confidence intervals

With about 137 effective shocks, the grid-inverted intervals are almost identical to the asymptotic ones.

In [7]:
ci = shiftshare.null_imposed_ci(industries, config.OUTCOME_POST, "tariff_shock", config.INDUSTRY_CONTROLS)
{k: ci[k] for k in ["estimate", "asymptotic_ci", "null_imposed_ci"]}

{'estimate': np.float64(3.1252208139295465),
 'asymptotic_ci': (-79.65016831997637, 85.90060994783546),
 'null_imposed_ci': (np.float64(-79.31179520676358),
  np.float64(85.5622368346227))}

## 7. Pre-trend falsification

The 2018 tariff shock does not predict 2014–2017 patenting (below) or 2013–2017 import trends (see
`outputs/tables/pre_trend_trade_regression.tex`).

In [8]:
pre = reg.innovation_pretrend(industries)
pre.params["tariff_shock"], pre.pvalues["tariff_shock"]

(np.float64(-4.86727116075942), np.float64(0.6035428342259426))

## Conclusion

Neither U.S. protective tariffs nor foreign retaliation had a statistically detectable short-run effect on local
patenting, but the confidence intervals are wide. See the [dissertation](../paper/Thesis.pdf) for the full discussion.